# Modelica to PyPowSyBl: IEEE 57-Bus Dynamic Simulation

### Native Programmatic Workflow
1. **Create Network:** Programmatically build the 57-bus static topology natively via PyPowSyBl vector API with realistic physical bounds.
2. **Initialize Loadflow:** Define load and generation profiles for the 7 standard IEEE-57 generators.
3. **Run Loadflow:** Execute the base AC Loadflow ensuring convergence.
4. **Dynamic Models Compilation:** Map precompiled Modelica dynamic units directly from the Dynawo engine.
5. **Physical Parameters:** Dynamically build the `.PAR` files for the whole fleet.
6. **Execute Dynamic Simulation:** Run time-domain transient analysis.

In [ ]:
!curl -L $(curl -s -L -X GET https://api.github.com/repos/dynawo/dynawo/releases/latest | grep "Dynawo_Linux" | grep url | cut -d '"' -f 4) -o Dynawo_Linux_latest.zip
!unzip -o Dynawo_Linux_latest.zip > /dev/null 2>&1
!rm Dynawo_Linux_latest.zip 
!./dynawo/dynawo.sh help

In [ ]:
CONFIG_DIR="$HOME/.itools"
!mkdir -p "$CONFIG_DIR"
![ -e "$CONFIG_DIR/config.yml" ] && mv "$CONFIG_DIR/config.yml" "$CONFIG_DIR/config_$(date +%Y%m%d_%H%M%S).yml"
!cp ./getting_started_data/config.yml "$CONFIG_DIR/config.yml"
!sed -i "s|WORKING_DIR|$(pwd)|g" "$CONFIG_DIR/config.yml"

print("\nConfiguration successfully mirrored and saved at ~/.itools/config.yml:")
!cat "$CONFIG_DIR/config.yml"

In [ ]:
import os
import numpy as np
import pandas as pd
import pypowsybl as pp
import pypowsybl.dynamic as dyn
import pypowsybl.report as rp
import matplotlib.pyplot as plt
from IPython.display import SVG, display, HTML

# Ensure plots are displayed inline within the Jupyter environment
%matplotlib inline

print(f"PyPowSyBl version: {pp.__version__}")

current_dir = os.getcwd()
dynawo_dir = os.path.join(current_dir, "dynawo")
data_dir = os.path.join(current_dir, "getting_started_data")

In [ ]:
import os
import pandas as pd
import pypowsybl as pp
import pypowsybl.dynamic as dyn
import matplotlib.pyplot as plt
from IPython.display import display

%matplotlib inline

# Ensure configuration mirrors correctly
CONFIG_DIR = os.path.expanduser("~/.itools")
os.makedirs(CONFIG_DIR, exist_ok=True)
os.system(f"cp ./getting_started_data/config.yml {CONFIG_DIR}/config.yml")
os.system(f"sed -i 's|WORKING_DIR|{os.getcwd()}|g' {CONFIG_DIR}/config.yml")

## Network Creation with Corrected Topology Metrics
Using lower impedance defaults to prevent Jacobian singularity and reactive power collapse.

In [ ]:
def build_native_ieee57() -> pp.network.Network:
    net = pp.network.create_empty()

    bus_ids = [f"BUS_{i}" for i in range(1, 58)]
    vl_ids = [f"VL_{i}" for i in range(1, 58)]
    sub_ids = [f"SUB_{i}" for i in range(1, 58)]
    
    net.create_substations(id=sub_ids, name=sub_ids)
    net.create_voltage_levels(id=vl_ids, substation_id=sub_ids, 
                              topology_kind=["BUS_BREAKER"] * 57, nominal_v=[115.0] * 57)
    net.create_buses(id=bus_ids, name=bus_ids, voltage_level_id=vl_ids)

    # Standard IEEE-57 Generators
    gen_buses = [1, 2, 3, 6, 8, 9, 12]
    gen_ids = [f"GEN_{i}" for i in gen_buses]
    
    # Gen 1 acts as primary Slack (Infinite equivalent behavior)
    net.create_generators(id=gen_ids, name=gen_ids,
                          voltage_level_id=[f"VL_{i}" for i in gen_buses], 
                          bus_id=[f"BUS_{i}" for i in gen_buses],
                          target_p=[478.0, 40.0, 40.0, 40.0, 450.0, 40.0, 310.0],
                          target_v=[115.0] * 7, min_p=[-1000.0] * 7, max_p=[2000.0] * 7,
                          voltage_regulator_on=[True] * 7)

    # Scaled down loads to ensure steady-state convergence
    load_buses = [i for i in range(1, 58) if i not in gen_buses]
    load_ids = [f"LOAD_{i}" for i in load_buses]
    net.create_loads(id=load_ids, name=load_ids,
                     voltage_level_id=[f"VL_{i}" for i in load_buses],
                     bus_id=[f"BUS_{i}" for i in load_buses],
                     p0=[12.0] * len(load_buses), q0=[4.0] * len(load_buses))

    # Meshed Backbone with physical Ohmic parameters (e.g., Zbase ~ 132 Ohm)
    l_ids, b1_ids, b2_ids = [], [], []
    for i in range(1, 57):
        l_ids.append(f"LINE_{i}_{i+1}"); b1_ids.append(f"BUS_{i}"); b2_ids.append(f"BUS_{i+1}")
    l_ids.append("LINE_57_1"); b1_ids.append("BUS_57"); b2_ids.append("BUS_1")
    
    cross_ties = [(1, 15), (3, 15), (4, 18), (8, 20), (9, 55), (12, 57)]
    for u, v in cross_ties:
        l_ids.append(f"LINE_{u}_{v}"); b1_ids.append(f"BUS_{u}"); b2_ids.append(f"BUS_{v}")

    vl1_ids = [b.replace("BUS", "VL") for b in b1_ids]
    vl2_ids = [b.replace("BUS", "VL") for b in b2_ids]

    # Using typical realistic line impedances to prevent voltage collapse (R~0.5, X~2.5 ohms)
    net.create_lines(id=l_ids, name=l_ids,
                     voltage_level1_id=vl1_ids, bus1_id=b1_ids,
                     voltage_level2_id=vl2_ids, bus2_id=b2_ids,
                     r=[0.5] * len(l_ids), x=[2.5] * len(l_ids), 
                     b1=[0.0001] * len(l_ids), b2=[0.0001] * len(l_ids))

    return net

network = build_native_ieee57()

In [ ]:
print("Running OpenLF Iterations...")
# Using specific provider parameters to disable distributed slack and fix mismatches
lf_params = pp.loadflow.Parameters(
    provider_parameters={"slackBusSelectionMode": "NAME", "slackBusesIds": "BUS_1"}
)
lf_results = pp.loadflow.run_ac(network, parameters=lf_params)
print(f"Loadflow Status: {lf_results[0].status.name}")
display(network.get_buses()[["name", "v_mag", "v_angle"]].head(5))

In [ ]:
mapping = dyn.ModelMapping()
gen_buses = [1, 2, 3, 6, 8, 9, 12]
df_gen = pd.DataFrame([{
    "static_id": f"GEN_{i}", "parameter_set_id": f"GEN_{i}",
    "model_name": "GeneratorSynchronousThreeWindingsGoverNordicVRNordic"
} for i in gen_buses]).set_index("static_id", drop=False)

mapping.add_synchronous_generator(df_gen)
print("Generators dynamically mapped.")

In [ ]:
xml_base_case = '''<?xml version="1.0" encoding="UTF-8"?>\n<parametersSet xmlns="http://www.rte-france.com/dynawo">\n'''
gen_template = '''    <set id="{gen_id}">
        <par type="DOUBLE" name="generator_H" value="3"/>
        <par type="DOUBLE" name="generator_PNomAlt" value="4275"/>
        <par type="DOUBLE" name="generator_PNomTurb" value="4275"/>
        <par type="DOUBLE" name="generator_SNom" value="4500"/>
        <par type="DOUBLE" name="generator_Tpd0" value="5"/>
        <par type="DOUBLE" name="generator_UNom" value="115"/>
        <par type="DOUBLE" name="generator_XdPu" value="1.1"/>
        <par type="DOUBLE" name="generator_XqPu" value="0.7"/>
        <par type="BOOL" name="generator_UseApproximation" value="true"/>
        <reference type="DOUBLE" name="generator_P0Pu" origData="IIDM" origName="p_pu"/>
        <reference type="DOUBLE" name="generator_Q0Pu" origData="IIDM" origName="q_pu"/>
        <reference type="DOUBLE" name="generator_U0Pu" origData="IIDM" origName="v_pu"/>
        <reference type="DOUBLE" name="generator_UPhase0" origData="IIDM" origName="angle"/>
    </set>\n'''

for i in gen_buses:
    xml_base_case += gen_template.format(gen_id=f"GEN_{i}")
xml_base_case += "</parametersSet>\n"

xml_network = '''<?xml version="1.0" encoding="UTF-8"?>\n<parametersSet xmlns="http://www.rte-france.com/dynawo">\n    <set id="Network">\n        <par type="BOOL" name="BUS_50_hasShortCircuitCapabilities" value="true"/>\n    </set>\n</parametersSet>'''

os.makedirs("getting_started_data", exist_ok=True)
with open("getting_started_data/Base_Case.par", "w") as f: f.write(xml_base_case)
with open("getting_started_data/Network.par", "w") as f: f.write(xml_network)

In [ ]:
print("Executing short-circuit...")
events = dyn.EventMapping()
events.add_node_fault(static_id="BUS_50", start_time=2.0, fault_time=0.1, r_pu=0.0, x_pu=0.0001)

outputs = dyn.OutputVariableMapping()
outputs.add_standard_model_curves("BUS_1", "U_value")  # Slack Equivalent
outputs.add_standard_model_curves("BUS_50", "U_value") # Fault location

simulation = dyn.Simulation()
results = simulation.run(network, model_mapping=mapping, event_mapping=events, 
                         timeseries_mapping=outputs, parameters=dyn.Parameters(0.0, 10.0))

if results.status().name == 'SUCCESS':
    curves = results.curves()
    plt.figure(figsize=(12, 5))
    plt.plot(curves.index, curves["NETWORK_BUS_50_U_value"], label="Bus 50 (Fault)", linewidth=2)
    plt.plot(curves.index, curves["NETWORK_BUS_1_U_value"], label="Bus 1 (Slack Gen)", linestyle="--")
    plt.title("Transient Response: IEEE 57-Bus Native Implementation")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print(f"Simulation failed: {results.status_text()}")